# BioAge-X: End-to-End Biological Age Estimation Quickstart

This tutorial demonstrates how to:
1. Ingest and profile a synthetic multi-omics cohort.
2. Preprocess DNA methylation and transcriptomics features.
3. Train and compare ElasticNet, Random Forest, and XGBoost models.
4. Calculate biological age acceleration residuals.
5. Explain predictions using SHAP and construct a biological interaction network.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from bioage.ingestion.loaders import DatasetLoader
from bioage.preprocessing.methylation import MethylationPreprocessor
from bioage.preprocessing.transcriptomics import TranscriptomicsPreprocessor
from bioage.preprocessing.feature_selection import FeatureSelector
from bioage.models.elasticnet import BioAgeElasticNet
from bioage.models.xgboost_model import BioAgeXGBoost
from bioage.evaluation.metrics import evaluate_predictions
from bioage.evaluation.acceleration import compute_age_acceleration
from bioage.explainability.shap_engine import BioAgeShapExplainer
from bioage.network.interaction_graph import BiologicalInteractionGraph

## 1. Load and Profile Demo Multi-Omics Cohort

In [ ]:
loader = DatasetLoader()
df, profile = loader.load_file("../data/example/demo_multiomics.csv")
print(f"Samples: {profile.n_samples}, Features: {profile.n_features}, Modality: {profile.detected_modality}")

## 2. Train XGBoost Biological Age Predictor

In [ ]:
y = df["chronological_age"]
drop_cols = {"sample_id", "chronological_age", "sex", "smoking_status", "bmi", "true_age_acceleration", "data_provenance"}
feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols]

selector = FeatureSelector(max_features=30, method="mutual_info")
X_sel = selector.fit_transform(X, y)

model = BioAgeXGBoost(n_estimators=50, max_depth=3)
model.fit(X_sel, y)
preds = model.predict(X_sel)

metrics = evaluate_predictions(y.values, preds, n_features=30)
print(f"MAE: {metrics.mae:.2f} years, R2: {metrics.r2:.3f}, Pearson r: {metrics.pearson_r:.3f}")

## 3. Calculate Age Acceleration

In [ ]:
df_accel = compute_age_acceleration(y, preds, sample_ids=list(df.index))
df_accel.head()

## 4. SHAP Attribution & Network Construction

In [ ]:
explainer = BioAgeShapExplainer(model)
explainer.explain(X_sel, sample_ids=list(df.index))
top_biomarkers = explainer.get_global_importance(top_k=10)
print("Top Biomarkers:")
for b in top_biomarkers[:5]:
    print(f" - {b['gene_symbol']}: mean |SHAP| = {b['mean_abs_shap']:.3f} ({b['direction']})")

# Build Network
graph = BiologicalInteractionGraph()
graph.build_from_biomarkers([b['gene_symbol'] for b in top_biomarkers], edge_list_path="../data/example/aging_network_edges.csv")
print(f"Constructed Graph: {graph.graph.number_of_nodes()} nodes, {graph.graph.number_of_edges()} edges")